<a href="https://colab.research.google.com/github/sowonjeong/slm-2-llm/blob/main/federalist_generate_llm_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
cd /content/drive/MyDrive/colab-github/txt-analysis

/content/drive/MyDrive/colab-github/txt-analysis


In [ ]:
# Load dataset from syllogi, no modification made
import json
with open("federalist.json") as f:
    federalist_dict = json.load(f)

In [ ]:
import pandas as pd

# Initialize an empty list to store the flattened data
flattened_data = []

# Iterate over each dictionary in the list
for item in federalist_dict:
    # Extract metadata and content
    meta = item['meta'][0]  # Assuming there's only one meta per item
    paper = item['paper'][0] if item['paper'] else ''  # Assuming content is a list

    # Flatten the data
    flattened_data.append({
        'meta_number': meta.get('number', ''),
        'meta_author': meta.get('author', ''),
        'meta_title': meta.get('title', ''),
        'meta_journal': meta.get('journal', ''),
        'meta_body': paper
    })

# Create a DataFrame
df = pd.DataFrame(flattened_data)
federalist_df = df.drop(df.index[69])

In [ ]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
from nltk.tokenize import sent_tokenize, word_tokenize
federalist_df['meta_sentence'] = federalist_df['meta_body'].apply(sent_tokenize)
federalist_df['meta_words'] = federalist_df['meta_sentence'].apply(lambda sentences: [word_tokenize(sentence) for sentence in sentences])

In [ ]:
import warnings
import pandas as pd
from nltk.tokenize import word_tokenize

# Suppress specific FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Assuming federalist_df is defined and has the required columns

# Create a list to store sentence-level data
sentence_data = []

# Iterate over each row in the original DataFrame
for index, row in federalist_df.iterrows():
    for i, sentence in enumerate(row['meta_sentence']):
        # Count the number of words in the sentence
        word_count = len(word_tokenize(sentence))

        # Append the data to the sentence_data list
        sentence_data.append({'meta_number': row['meta_number'],
                              'meta_author': row['meta_author'],
                              'meta_sentence_number': i + 1,
                              'meta_sentence': sentence,
                              'meta_num_of_words': word_count})

# Convert the list of dictionaries to a DataFrame
sentence_df = pd.DataFrame(sentence_data)

sentence_df


AttributeError: 'DataFrame' object has no attribute 'append'

In [ ]:
sentence_df['meta_sentence']

Series([], Name: meta_sentence, dtype: object)

In [ ]:
from sentence_transformers import SentenceTransformer
sent_model = SentenceTransformer("multi-qa-distilbert-cos-v1")

# Try chunking Text Instead of Parsing all Sentences

In [ ]:
sentence_df.groupby("meta_number")["meta_sentence_number"].max().reset_index()

In [ ]:
sentence_df

In [ ]:
import matplotlib.pyplot as plt
plt.hist(sentence_df['meta_num_of_words'])

In [ ]:
import re

def overlapping_chunks(text, max_tokens = 200, overlapping_factor = 5):
  '''
  max tokens: tokens we want per chunk
  overlapping_factor: number of sentences to start each chunk with that overlaps with the previous chunk
  '''

  # Split the text using puncutation
  sentences = re.split(r'[.?!]', text)

  # Get the number of tokens for each sentence
  n_tokens = [len(sentence.split()) for sentence in sentences]

  chunks, tokens_so_far, chunk = [], 0, []

  # Loop through the sentences and tokens joined together in tuple
  for sentence, token in zip(sentences, n_tokens):

    # If the number of tokens far plus the number of tokens in the current sentence is greater than the max number of tokens,
    # then add the chunk to the list of chunks and reset
    # the chunk and tokens so far
    if tokens_so_far + token > max_tokens:
      chunks.append(".".join(chunk)+".")
      if overlapping_factor>0:
        chunk = chunk[-overlapping_factor:]
        tokens_so_far = sum([len(c.split()) for c in chunk])
      else:
        chunk = []
        tokens_so_far = 0
      # If the number of tokens in the current sentence is greater than the max number of tokens,
      # go to the next sentence
    if token > max_tokens:
      continue

    # Otherwise, add the sentence to the chunk and add the number of tokens to the toal
    chunk.append(sentence)
    tokens_so_far += token + 1
  return chunks

In [ ]:
split = overlapping_chunks(federalist_df['meta_body'][0], overlapping_factor = 0)

In [ ]:
split[0]

In [ ]:
split[1]

In [ ]:
split[2]

In [ ]:
avg_length = sum([len(tokenizer.encode(t)) for t in split])/len(split)
print(f'non-overlapping chunking approach has {len(split)} documents with average length {avg_length: .1f} tokens.')

In [ ]:
# with overlapping chunk
split = overlapping_chunks(federalist_df['meta_body'][0], overlapping_factor = 5)
avg_length = sum([len(tokenizer.encode(t)) for t in split])/len(split)
print(f'overlapping chunking approach has {len(split)} documents with average length {avg_length: .1f} tokens.')

In [ ]:
federalist_df = federalist_df.reset_index()

In [ ]:
authors = federalist_df['meta_author']

In [ ]:
authors

In [ ]:
import pandas as pd
import numpy as np

tlength = [50, 100, 150, 200, 250, 300]
for t in tlength:
  chunking_data = []  # This will store the flat list of [chunk, author] pairs

  for i in np.arange(85):
      # Call your function to split text into chunks
      splits = overlapping_chunks(federalist_df['meta_body'][i], max_tokens = t, overlapping_factor=0)
      # Get the author for all chunks of this text
      author = authors[i]
      # Extend the list with pairs of [chunk, author]
      chunking_data.extend([[split, author, i+1 ] for split in splits])

  # Create a DataFrame from the list of [chunk, author] pairs
  chunking_df = pd.DataFrame(chunking_data, columns=['split', 'author','doc_num'])
  answer_name = "chunking_author_"+str(t)+".csv"
  chunking_df[['author','doc_num']].to_csv(answer_name,index = False)
  chunk_embedding = sent_model.encode(
    list(chunking_df['split']),
    batch_size = 32,
    show_progress_bar = True)
  df_name= "chunk_embedding_"+str(t)+".csv"
  np.savetxt(df_name, chunk_embedding, delimiter=",", fmt='%.32f')


In [ ]:
import numpy as np


# Initialize an empty list to store the average document embeddings
average_chunk_embeddings = []

# Iterate over the indicator vectors to compute the average document embedding for each document
for indicator_vector in np.arange(1,86):
    # Compute the mask for the current document
    mask = chunking_df['doc_num'] == indicator_vector

    # Use the mask to select the rows corresponding to the current document
    chunk_embeddings_for_document = chunk_embedding[mask]

    # Compute the average document embedding for the current document
    average_chunk_embedding = np.mean(chunk_embeddings_for_document, axis=0)

    # Append the average document embedding to the list
    average_chunk_embeddings.append(average_chunk_embedding)

# Convert the list of average document embeddings to a numpy array
average_chunk_embeddings = np.array(average_chunk_embeddings)


In [ ]:
np.savetxt("avg_chunk_embedding.csv", average_chunk_embeddings, delimiter=",", fmt='%.32f')

In [ ]:
avg_length = sum([len(tokenizer.encode(t)) for t in list(chunking_df['split'])])/len(chunking_df['split'])
# print(f'non-overlapping chunking approach has {len(chunking_df['split'])} documents with average length {avg_length: .1f} tokens.')

In [ ]:
avg_length

In [ ]:
len(chunking_df['split'])

In [ ]:
chunking_df

In [ ]:
np.sum(chunking_df['author'] == 'HAMILTON OR MADISON')

In [ ]:
chunk_embedding = sent_model.encode(
    list(chunking_df['split']),
    batch_size = 32,
    show_progress_bar = True
)

In [ ]:
chunking_df['author'].to_csv("chunking_author.csv",index = False)

In [ ]:
np.savetxt("chunk_embedding.csv", chunk_embedding, delimiter=",", fmt='%.32f')

In [ ]:
chunk_embedding

In [ ]:
sentence_df[['meta_author','meta_number']].to_csv("sentence_author.csv",index = False)

In [ ]:
np.savetxt("sent_embedding.csv", doc_embedding, delimiter=",", fmt='%.32f')

In [ ]:
doc_embedding.shape

In [ ]:
import numpy as np


# Initialize an empty list to store the average document embeddings
average_doc_embeddings = []

# Iterate over the indicator vectors to compute the average document embedding for each document
for indicator_vector in np.arange(1,86):
    # Compute the mask for the current document
    mask = sentence_df['meta_number'] == indicator_vector

    # Use the mask to select the rows corresponding to the current document
    doc_embeddings_for_document = doc_embedding[mask]

    # Compute the average document embedding for the current document
    average_doc_embedding = np.mean(doc_embeddings_for_document, axis=0)

    # Append the average document embedding to the list
    average_doc_embeddings.append(average_doc_embedding)

# Convert the list of average document embeddings to a numpy array
average_doc_embeddings = np.array(average_doc_embeddings)


In [ ]:
np.savetxt("avg_sent_embedding.csv", average_doc_embeddings, delimiter=",", fmt='%.32f')

# WORD2VEC

In [ ]:
# !pip install gensim

In [ ]:
sentence_df['meta_sentence']

In [ ]:
tokenized = [word_tokenize(sentence) for sentence in sentence_df['meta_sentence']]

In [ ]:
len([word_tokenize(sentence) for sentence in sentence_df[sentence_df['meta_number']==1]['meta_sentence']])

In [ ]:
from gensim.models import KeyedVectors

# Load the model (This can take some time and memory)
model_path = 'GoogleNews-vectors-negative300.bin'
model = KeyedVectors.load_word2vec_format(model_path, binary=True)


In [ ]:
import os
import pandas as pd
from time import time
from nltk.tokenize import RegexpTokenizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

In [ ]:
# Define a custom tokenizer that filters out tokens containing only digits
def custom_tokenizer(text):
    tokens = word_tokenize(text)
    return [token for token in tokens if not re.match(r'^\d+$', token)]


# Use tf (raw term count) features for LDA.
print("Extracting tf features for LDA...")
tf_vectorizer = CountVectorizer(
    lowercase = True,
    # max_df=0.95, min_df=2,
    #max_features=2000,
    #stop_words="english",
    # tokenizer=custom_tokenizer,
    ngram_range = (1,1)
)
t0 = time()
tf = tf_vectorizer.fit_transform(federalist_df['meta_body'])
print("done in %0.3fs." % (time() - t0))
print()

Extracting tf features for LDA...
done in 0.333s.



In [ ]:
tf.shape

(85, 8653)

In [ ]:
list(tf_vectorizer.get_feature_names_out())
feature_names = list(tf_vectorizer.get_feature_names_out())
index_of_abandon = feature_names.index('abandon')
selected_features = feature_names[index_of_abandon:]
print(selected_features)

['abandon', 'abandoned', 'abandoning', 'abate', 'abatements', 'abb', 'abbe', 'abbé', 'abetted', 'abhorrence', 'abilities', 'ability', 'abject', 'able', 'ablest', 'abolish', 'abolished', 'abolishing', 'abolition', 'abortive', 'abounding', 'abounds', 'about', 'above', 'abraham', 'abridge', 'abridged', 'abridgements', 'abridging', 'abridgment', 'abroad', 'abrogate', 'abrogating', 'abrég', 'absence', 'absolute', 'absolutely', 'absolves', 'absorb', 'absorbed', 'abstain', 'abstained', 'abstract', 'abstracted', 'abstraction', 'abstruse', 'absurd', 'absurdities', 'absurdity', 'absurdly', 'abundance', 'abundant', 'abundantly', 'abuse', 'abused', 'abuses', 'abyss', 'accede', 'accelerating', 'accept', 'acceptable', 'acceptance', 'accepted', 'access', 'accessible', 'accession', 'accident', 'accidental', 'accommodate', 'accommodated', 'accommodation', 'accommodations', 'accomodating', 'accompanied', 'accompany', 'accomplice', 'accomplices', 'accomplish', 'accomplished', 'accomplishes', 'accomplishi

In [ ]:
len(selected_features)

8609

In [ ]:
selected_indices = [tf_vectorizer.vocabulary_[feature] for feature in selected_features]
tf[:, selected_indices]

<85x8609 sparse matrix of type '<class 'numpy.int64'>'
	with 59526 stored elements in Compressed Sparse Row format>

In [ ]:
words_not_in_model = [word for word in selected_features if word not in model]
words_in_model = [word for word in selected_features if word in model]

print(words_not_in_model)

['abb', 'abbé', 'abrég', 'achaean', 'achaeans', 'achaeus', 'achaia', 'admiralties', 'adulator', 'adversed', 'aetolians', 'allbe', 'allemagne', 'amadeus', 'amphictyon', 'amphictyonic', 'amphictyons', 'and', 'animadversion', 'annapolis', 'antiquaries', 'antirepublican', 'anycounterbalancing', 'apothegm', 'appertain', 'aragon', 'aratus', 'asiatic', 'aspasia', 'athenian', 'athenians', 'aulic', 'authoruze', 'bailages', 'bashaws', 'bavaria', 'belgic', 'berne', 'bils', 'borderers', 'brutus', 'bypaths', 'callicrates', 'cambray', 'carthage', 'catalogue', 'centre', 'centuriata', 'champlain', 'charlemagne', 'charta', 'chronol', 'clamour', 'clanship', 'cleomenes', 'cognizances', 'comitatus', 'comitia', 'concentred', 'confedracy', 'contradistinguished', 'contumacy', 'conveniency', 'copiousness', 'cosmi', 'counsellors', 'counterpoises', 'criminate', 'criticised', 'cromwell', 'culumniated', 'decemvirs', 'deleware', 'delphos', 'demesnes', 'demosthenes', 'dependants', 'descanted', 'descendible', 'dicti

In [ ]:
len(words_not_in_model)

339

In [ ]:
len(words_in_model)

8270

In [ ]:
selected_indices = [tf_vectorizer.vocabulary_[feature] for feature in words_in_model]
tf[:, selected_indices]

<85x8270 sparse matrix of type '<class 'numpy.int64'>'
	with 58688 stored elements in Compressed Sparse Row format>

In [ ]:
selected_embeddings = model[words_in_model]

In [ ]:
selected_embeddings.shape

(8270, 300)

In [ ]:
import numpy as np

count_matrix = tf[:, selected_indices].toarray()

# Compute weighted sum for each document
weighted_sums = np.dot(count_matrix, selected_embeddings)

# Compute total count for each document
total_counts = np.sum(count_matrix, axis=1, keepdims=True)

# Compute weighted average for each document
weighted_average_embeddings = weighted_sums / total_counts


In [ ]:
weighted_average_embeddings.shape

In [ ]:
weighted_average_embeddings

In [ ]:
words_in_model

In [ ]:
np.savetxt("weighted_word_embeddings.csv", weighted_average_embeddings, delimiter=",", fmt='%.32f')

In [ ]:
import numpy as np

word_vectors = {}
word_counts = {}

for sentence in tokenized:  # Assuming 'sentences' is your list of lists of words
    for word in sentence:
        if word in model:  # Check if the word is in the model
            if word in word_vectors:
                # If the word vector is already recorded, increment the count
                word_counts[word] += 1
            else:
                # Record the word vector and initialize the count
                word_vectors[word] = model[word]
                word_counts[word] = 1


In [ ]:
pd.DataFrame.from_dict(word_counts, orient = 'index').to_csv("word_counts.csv")

In [ ]:
pd.DataFrame.from_dict(word_vectors, orient = 'index').to_csv("word_vectors.csv")

In [ ]:
word_vectors['majority'].shape